# 🎤 STT avec Faster-Whisper

Ce notebook permet d'expérimenter avec **faster-whisper** pour la transcription audio.

## Installation

In [1]:
# Installation des dépendances
!pip install faster-whisper

  Using cached pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)
  Using cached filelock-3.20.3-py3-none-any.whl.metadata (2.1 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached certifi-2026.1.4-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 18.2 MB/s  0:00:00
   ---------------------------------------- 0.0/18.6 MB ? eta -:--:--
   ----------------------- ---------------- 11.0/18.6 MB 57.2 MB/s eta 0:00:01
   ---------------------------------------- 18.6/18.6 MB 51.0 MB/s  0:00:00
   ---------------------------------------- 0.0/13.5 MB ? eta -:--:--
   -------------------------------------- - 12.8/13.5 MB 66.9 MB/s eta 0:00:01
   ---------------------------------------- 13.5/13.5 MB 49.7 MB/s  0:00:00
Usi

## Configuration du modèle

Modèles disponibles (du plus léger au plus précis) :
- `tiny` (~75 Mo) - très rapide, moins précis
- `base` (~145 Mo) - bon compromis pour un PoC
- `small` (~488 Mo) - meilleure qualité
- `medium` (~1.5 Go) - très bonne qualité
- `large-v3` (~3 Go) - meilleure qualité possible

In [8]:
from faster_whisper import WhisperModel

# Configuration
MODEL_SIZE = "small"  # Changer selon tes besoins
DEVICE = "cpu"       # "cuda" si tu as un GPU NVIDIA
COMPUTE_TYPE = "int8"  # "float16" pour GPU, "int8" pour CPU

print(f"Chargement du modèle {MODEL_SIZE}...")
model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)
print("Modèle chargé !")

Chargement du modèle small...


c:\Users\yassi\Desktop\projects\station-ia-embeded\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yassi\.cache\huggingface\hub\models--Systran--faster-whisper-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Modèle chargé !


## Test avec un fichier audio

Place un fichier audio (WAV, MP3, etc.) dans le dossier `ia/` ou indique le chemin complet.

In [9]:
import time

# Chemin vers ton fichier audio de test
AUDIO_FILE = "test2.m4a"  # À modifier selon ton fichier

# Transcription
print(f"Transcription de {AUDIO_FILE}...")
start_time = time.time()

segments, info = model.transcribe(
    AUDIO_FILE,
    language="fr",  # Force le français
    beam_size=5
)

# Récupération du texte complet
transcription = " ".join([segment.text for segment in segments])

elapsed_time = time.time() - start_time

print(f"\n📊 Résultats :")
print(f"  - Langue détectée : {info.language}")
print(f"  - Probabilité langue : {info.language_probability:.2%}")
print(f"  - Durée audio : {info.duration:.2f}s")
print(f"  - Temps de traitement : {elapsed_time:.2f}s")
print(f"  - Ratio (temps réel) : {elapsed_time / info.duration:.2f}x")
print(f"\n📝 Transcription :\n{transcription}")

Transcription de test2.m4a...

📊 Résultats :
  - Langue détectée : fr
  - Probabilité langue : 100.00%
  - Durée audio : 5.99s
  - Temps de traitement : 2.82s
  - Ratio (temps réel) : 0.47x

📝 Transcription :
 bonjour la Terre, test 1, 2, test


## Test avec enregistrement micro (optionnel)

Si tu veux tester directement avec ton micro.

In [10]:
# Installation pour l'enregistrement audio
!pip install sounddevice soundfile

  Using cached cffi-2.0.0-cp312-cp312-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 16.1 MB/s  0:00:00
Using cached cffi-2.0.0-cp312-cp312-win_amd64.whl (183 kB)
Using cached pycparser-3.0-py3-none-any.whl (48 kB)

   ---------------------------------------- 0/4 [pycparser]
   ---------- ----------------------------- 1/4 [cffi]
   ---------- ----------------------------- 1/4 [cffi]
   ------------------------------ --------- 3/4 [sounddevice]
   ---------------------------------------- 4/4 [sounddevice]



In [13]:
import sounddevice as sd
import soundfile as sf
import numpy as np

# Configuration de l'enregistrement
SAMPLE_RATE = 16000  # 16kHz comme recommandé
DURATION = 5  # Durée en secondes

print(f"🎙️ Enregistrement pendant {DURATION} secondes...")
print("Parle maintenant !")

# Enregistrement
audio = sd.rec(int(DURATION * SAMPLE_RATE), samplerate=SAMPLE_RATE, channels=1, dtype='int16')
sd.wait()  # Attend la fin de l'enregistrement

# Sauvegarde temporaire
temp_file = "temp_recording.wav"
sf.write(temp_file, audio, SAMPLE_RATE)
print(f"✅ Enregistrement sauvegardé dans {temp_file}")

🎙️ Enregistrement pendant 5 secondes...
Parle maintenant !
✅ Enregistrement sauvegardé dans temp_recording.wav


In [14]:
# Transcription de l'enregistrement
import time

print("Transcription en cours...")
start_time = time.time()

segments, info = model.transcribe("temp_recording.wav", language="fr")
transcription = " ".join([segment.text for segment in segments])

elapsed_time = time.time() - start_time

print(f"\n⏱️ Temps de traitement : {elapsed_time:.2f}s")
print(f"📝 Transcription : {transcription}")

Transcription en cours...

⏱️ Temps de traitement : 2.68s
📝 Transcription :  Bonjour à tous, c'est moi, et venez !


## Fonction utilitaire pour l'API

Voici une fonction prête à être utilisée dans l'API FastAPI.

In [ ]:
from faster_whisper import WhisperModel
from typing import Tuple
import io

class STTService:
    """Service de Speech-to-Text avec Faster-Whisper"""
    
    def __init__(self, model_size: str = "base", device: str = "cpu"):
        compute_type = "float16" if device == "cuda" else "int8"
        self.model = WhisperModel(model_size, device=device, compute_type=compute_type)
    
    def transcribe(self, audio_path: str, language: str = "fr") -> Tuple[str, dict]:
        """
        Transcrit un fichier audio en texte.
        
        Args:
            audio_path: Chemin vers le fichier audio
            language: Code langue (fr, en, etc.)
            
        Returns:
            Tuple (texte, métadonnées)
        """
        segments, info = self.model.transcribe(audio_path, language=language)
        text = " ".join([s.text for s in segments]).strip()
        
        metadata = {
            "language": info.language,
            "language_probability": info.language_probability,
            "duration": info.duration
        }
        
        return text, metadata

# Test
# stt = STTService(model_size="base")
# text, meta = stt.transcribe("test.wav")
# print(text, meta)

## Notes

- Le premier chargement du modèle télécharge les poids (~145 Mo pour `base`)
- Les poids sont cachés dans `~/.cache/huggingface/`
- Sur CPU, le modèle `base` traite ~0.5x temps réel (5s audio = ~10s traitement)
- Sur GPU CUDA, c'est ~10x plus rapide